In [1]:
!nvidia-smi

Mon Jun 15 14:37:09 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 581.57                 Driver Version: 581.57         CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                  Driver-Model | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 5060 Ti   WDDM  |   00000000:01:00.0  On |                  N/A |
|  0%   43C    P8             12W /  180W |    1441MiB /  16311MiB |      5%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!pip install transformers datasets peft bitsandbytes pymatgen pandas numpy matplotlib tqdm


[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
!pip install transformers==4.53.0 accelerate peft tokenizers -U

  Using cached tokenizers-0.23.1-cp310-abi3-win_amd64.whl.metadata (10 kB)
   ---------------------------------------- 0.0/389.2 kB ? eta -:--:--
   - -------------------------------------- 10.2/389.2 kB ? eta -:--:--
   --- ----------------------------------- 30.7/389.2 kB 660.6 kB/s eta 0:00:01
   --------- ----------------------------- 92.2/389.2 kB 871.5 kB/s eta 0:00:01
   --------------- ------------------------ 153.6/389.2 kB 1.0 MB/s eta 0:00:01
   ------------------------ --------------- 235.5/389.2 kB 1.3 MB/s eta 0:00:01
   ---------------------------------------- 389.2/389.2 kB 1.6 MB/s eta 0:00:00
  Attempting uninstall: accelerate
    Found existing installation: accelerate 1.13.0
    Uninstalling accelerate-1.13.0:
      Successfully uninstalled accelerate-1.13.0



[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
# !pip install transformers==4.41.2
# !pip install peft==0.11.1
# !pip install accelerate==0.31.0
# !pip install trl==0.9.4
# !pip install bitsandbytes

In [5]:
import torch, gc

# Delete old model and free CUDA memory
#del model
gc.collect()
torch.cuda.empty_cache()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("CUDA available:", torch.cuda.is_available())
print("Device:", device)


CUDA available: True
Device: cuda


In [6]:
import pandas as pd
from sklearn.model_selection import train_test_split

CSV_PATH = r"C:\Users\Admin\Downloads\combined\combined data TE\combined_all_data_less_than_40.csv"
df = pd.read_csv(CSV_PATH)

df = df.dropna(subset=["Formula","Temperature (K)","TC"]).copy()
df["Formula"] = df["Formula"].astype(str)
df["Temperature (K)"] = df["Temperature (K)"].astype(str)
df["TC"] = df["TC"].astype(float)

train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)

print(len(train_df), len(test_df))
train_df.head()


10478 2620


,Formula,Temperature (K),TC,DOI
5758,Ba14.9472Cu9.05723Si75.9956,713.0,2.800,10.1103/physrevb.87.115206
4244,Nb28.33Ti1V1Hf1Mo1Zr1Fe33.33Sb33.33,307.0,8.260,NaN
7757,Ag25Bi25Se50,600.0,0.410,https://doi.org/10.1021/acs.inorgchem.9b00874
415,Pb84.48Na0.25S15.27,450.0,1.640,NaN
11444,Co25Sb75,772.26,3.087,NaN


In [7]:
def build_tg_example(row, unit="W/mK", temp_unit="K"):
    formula = row["Formula"]
    temperature = float(row["Temperature (K)"])
    TC = float(row["TC"])

    prompt = (
        "Instruction: Given the following material formula and temperature, "
        f"predict its thermal conductivity (TC) in {unit}.\n\n"
        f"[Formula] {formula}\n"
        f"[Temperature] {temperature} {temp_unit}\n\n"
        "TC:"
    )

    # Target: a single numerical value
    target = f" {TC:.4f}"
    return {"prompt": prompt, "target": target}


train_samples = [build_tg_example(r) for _, r in train_df.iterrows()]
test_samples  = [build_tg_example(r) for _, r in test_df.iterrows()]

print(train_samples[0]["prompt"])
print(train_samples[0]["target"])


Instruction: Given the following material formula and temperature, predict its thermal conductivity (TC) in W/mK.

[Formula] Ba14.9472Cu9.05723Si75.9956
[Temperature] 713.0 K

TC:
 2.8000


In [8]:
import json
from pathlib import Path
from datasets import load_dataset

PROJECT_ROOT = Path(".")
PROCESSED_DIR = PROJECT_ROOT / "processed_TC"
PROCESSED_DIR.mkdir(exist_ok=True)

def save_jsonl(samples, path):
    with path.open("w", encoding="utf-8") as f:
        for s in samples:
            f.write(json.dumps(s, ensure_ascii=False) + "\n")

save_jsonl(train_samples, PROCESSED_DIR / "train.jsonl")
save_jsonl(test_samples,  PROCESSED_DIR / "test.jsonl")

raw_datasets = load_dataset("json", data_files={
    "train": str(PROCESSED_DIR / "train.jsonl"),
    "test": str(PROCESSED_DIR / "test.jsonl"),
})
print(raw_datasets)


Generating train split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['prompt', 'target'],
        num_rows: 10478
    })
    test: Dataset({
        features: ['prompt', 'target'],
        num_rows: 2620
    })
})


In [9]:
!pip install sentencepiece protobuf


[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [10]:
from transformers import AutoTokenizer

model_name = "Qwen/Qwen2.5-7B"

tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    use_fast=False,
    trust_remote_code=True,
    legacy=False
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

MAX_LEN = 512  # Material formulas are typically shorter than CIF data; adjust this value as needed

def tokenize_example(example):
    prompt = example["prompt"]
    target = example["target"]
    full_text = prompt + target

    full_tok = tokenizer(
        full_text,
        truncation=True,
        max_length=MAX_LEN,
        padding=False,
        add_special_tokens=False,
    )

    prompt_tok = tokenizer(
        prompt,
        truncation=True,
        max_length=MAX_LEN,
        padding=False,
        add_special_tokens=False,
    )

    input_ids = full_tok["input_ids"]
    labels = input_ids.copy()

    prompt_len = len(prompt_tok["input_ids"])
    labels[:prompt_len] = [-100] * prompt_len

    full_tok["labels"] = labels
    return full_tok

tokenized_datasets = raw_datasets.map(
    tokenize_example,
    remove_columns=raw_datasets["train"].column_names
)
print(tokenized_datasets)


Map:   0%|          | 0/10478 [00:00<?, ? examples/s]

Map:   0%|          | 0/2620 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 10478
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 2620
    })
})


In [11]:
from dataclasses import dataclass
from typing import Any, Dict, List
import torch
from transformers import DataCollatorWithPadding

@dataclass
class DataCollatorForCausalLMWithLabels:
    tokenizer: Any

    def __call__(self, features: List[Dict[str, Any]]) -> Dict[str, Any]:
        # Extract labels from the input features
        labels = [f["labels"] for f in features]
        for f in features:
            f.pop("labels")

        # Pad input_ids and attention_mask within the batch
        batch = DataCollatorWithPadding(
            tokenizer=self.tokenizer,
            padding=True,
            return_tensors="pt"
        )(features)

        # Pad labels with -100 up to max_len
        max_len = batch["input_ids"].shape[1]
        padded_labels = []
        for l in labels:
            if len(l) > max_len:
                l = l[:max_len]
            padded_labels.append(l + [-100] * (max_len - len(l)))

        batch["labels"] = torch.tensor(padded_labels, dtype=torch.long)
        return batch

data_collator = DataCollatorForCausalLMWithLabels(tokenizer)


In [12]:
import torch
from pathlib import Path

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    AutoConfig,
)

from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training,
)


model_name = "Qwen/Qwen2.5-7B"

# =========================
# 2. Tokenizer
# =========================
tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    use_fast=False,
    trust_remote_code=True
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# =========================
# 3. Config
# =========================
config = AutoConfig.from_pretrained(
    model_name,
    trust_remote_code=True,
)

config.pad_token_id = tokenizer.pad_token_id
config.use_cache = False

from transformers import BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    config=config,
    quantization_config=bnb_config,
    device_map={"": 0},
    trust_remote_code=True,
)
# =========================
# 6. LoRA config
# =========================
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
)

model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()

# =========================
# 7. Training arguments
# =========================
output_dir = str(PROJECT_ROOT / "deepseek_formula_TC_lora")

training_args = TrainingArguments(
    output_dir=output_dir,

    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    num_train_epochs=3,

    learning_rate=2e-4,
    fp16=True,

    logging_steps=20,
    save_steps=200,
    save_total_limit=2,

    report_to=[],
    remove_unused_columns=False,
)

# =========================
# 8. Trainer
# =========================
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    data_collator=data_collator,
)

trainer.train()


W0615 14:38:13.057000 16848 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.
W0615 14:38:16.428000 16848 site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

c:\Users\Admin\AppData\Local\Programs\Python\Python311\Lib\site-packages\bitsandbytes\backends\cuda\ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


trainable params: 40,370,176 || all params: 7,655,986,688 || trainable%: 0.5273


Step,Training Loss
20,1.194000
40,1.080400
60,1.057600
80,1.047200
100,1.034200
120,1.045600
140,1.032000
160,0.994200
180,0.992700
200,1.015100


TrainOutput(global_step=1965, training_loss=0.8482551836785469, metrics={'train_runtime': 5229.3595, 'train_samples_per_second': 6.011, 'train_steps_per_second': 0.376, 'total_flos': 1.0020991024192512e+17, 'train_loss': 0.8482551836785469, 'epoch': 3.0})

In [13]:
import re

@torch.no_grad()
def predict_TC(Formula, temperature, unit="W/mK", temp_unit="K", max_new_tokens=16):
    prompt = (
        "Instruction: Given the following material formula and temperature, "
        f"predict its thermal conductivity (TC) in {unit}.\n\n"
        f"[Formula] {Formula}\n\n"
        f"[Temperature] {temperature} {temp_unit}\n\n"
        "TC:"
    )

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    out = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )

    text = tokenizer.decode(out[0], skip_special_tokens=True)

    # Extract the text after TC:
    pred_txt = text.split("TC:")[-1].strip()

    # Parse a number (supports formats such as -123.45 and 1e3)
    m = re.search(r"[-+]?\d*\.?\d+(?:[eE][-+]?\d+)?", pred_txt)
    if not m:
        return None, pred_txt  # Also return the raw text for debugging
    return float(m.group(0)), pred_txt

TC_pred, raw = predict_TC("Cu50Zr50", "500")  # Example
print("Pred TC:", TC_pred, "| raw:", raw)


c:\Users\Admin\AppData\Local\Programs\Python\Python311\Lib\site-packages\bitsandbytes\backends\cuda\ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Pred TC: 0.2 | raw: 0.2000000000000


In [14]:
import numpy as np
from sklearn.metrics import r2_score

y_true, y_pred = [], []
for _, r in test_df.iterrows():
    pred, _ = predict_TC(r["Formula"], r["Temperature (K)"])
    if pred is None:
        continue
    y_true.append(float(r["TC"]))
    y_pred.append(pred)

y_true = np.array(y_true)
y_pred = np.array(y_pred)

mae = np.mean(np.abs(y_true - y_pred))
rmse = np.sqrt(np.mean((y_true - y_pred)**2))

r2 = r2_score(y_true, y_pred)

print("Test MAE:", mae)
print("Test RMSE:", rmse)
print("Test R2: ", r2)


c:\Users\Admin\AppData\Local\Programs\Python\Python311\Lib\site-packages\bitsandbytes\backends\cuda\ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Test MAE: 0.16101162330657412
Test RMSE: 0.43119395697787305
Test R2:  0.9750372414637558


In [15]:
import pandas as pd
import re
import torch
from tqdm import tqdm

# =========================
# 1. Path to the input file for prediction
# =========================
INPUT_CSV = r"C:\Users\Admin\Downloads\combined\combined data TE\data_validate_TC_thermoelectric.csv"
OUTPUT_CSV = r"data_validate_TC_thermoelectric_results.csv"

df_pred = pd.read_csv(INPUT_CSV)

# Check the required columns
required_cols = ["Formula", "Temperature (K)"]
for col in required_cols:
    if col not in df_pred.columns:
        raise ValueError(f"Missing column: {col}")

df_pred = df_pred.dropna(subset=["Formula", "Temperature (K)"]).copy()
df_pred["Formula"] = df_pred["Formula"].astype(str)
df_pred["Temperature (K)"] = df_pred["Temperature (K)"].astype(float)

# =========================
# 2. TC prediction function
# =========================
@torch.no_grad()
def predict_TC(formula, temperature, unit="W/mK", temp_unit="K", max_new_tokens=16):
    prompt = (
        "Instruction: Given the following material formula and temperature, "
        f"predict its thermal conductivity (TC) in {unit}.\n\n"
        f"[Formula] {formula}\n"
        f"[Temperature] {temperature} {temp_unit}\n\n"
        "TC:"
    )

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    out = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )

    text = tokenizer.decode(out[0], skip_special_tokens=True)
    pred_txt = text.split("TC:")[-1].strip()

    m = re.search(r"[-+]?\d*\.?\d+(?:[eE][-+]?\d+)?", pred_txt)
    if m:
        return float(m.group(0))
    else:
        return None

# =========================
# 3. Generate predictions for the entire file
# =========================
model.eval()

tc_preds = []

for _, row in tqdm(df_pred.iterrows(), total=len(df_pred)):
    pred = predict_TC(
        formula=row["Formula"],
        temperature=row["Temperature (K)"]
    )
    tc_preds.append(pred)

df_pred["TC_predicted"] = tc_preds

# =========================
# 4. Save the results
# =========================
df_pred.to_csv(OUTPUT_CSV, index=False)

print(df_pred.head())
print("Saved to:", OUTPUT_CSV)

  0%|          | 0/90 [00:00<?, ?it/s]c:\Users\Admin\AppData\Local\Programs\Python\Python311\Lib\site-packages\bitsandbytes\backends\cuda\ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
100%|██████████| 90/90 [02:38<00:00,  1.77s/it]

    Formula  Temperature (K)       TC  TC_predicted
0  Pb50Se50        303.55252  1.85714      2.014206
1  Pb50Se50        372.04027  1.57190      1.704888
2  Pb50Se50        472.52187  1.28834      1.360641
3  Pb50Se50        574.46712  1.20179      1.018889
4  Pb50Se50        676.39121  1.28591      0.861061
Saved to: data_validate_TC_thermoelectric_results.csv
